In [1]:
import pandas as pd
import numpy as np

In [2]:
def load_raw_data(filepath):
    raw_df = pd.read_excel(filepath, header = [0,1])
    return raw_df

In [3]:

df = load_raw_data('toy_equity_half_blank.xlsx')
print('RAWWWWWWW')
print(df.head())

RAWWWWWWW
  Unnamed: 0_level_0 TOY_EQUITY US Equity                                 
               Dates              PX_LAST PX_VOLUME PX_OPEN PX_LOW PX_HIGH
0           2/1/2023                  NaN       NaN     NaN    NaN     NaN
1           3/1/2023                60.34  760880.0   60.64  59.74   60.94
2           4/1/2023                54.49  134750.0   54.76  53.95   55.03
3           5/1/2023                75.92  828583.0   76.30  75.16   76.68
4           6/1/2023                90.41  292487.0   90.86  89.51   91.31


In [4]:
def clean_data_date(raw_df):
    """
    Input : raw DataFrame
    Output: cleaned DataFrame —
            missing values handled, duplicates removed,
            sorted chronologically, consistent schema enforced
    """
    #-----------------Date-------------------#
    ##create new column to test check which date cannot be parsed successfully
    raw_df[('real_date','real_date')] = pd.to_datetime(raw_df[('Unnamed: 0_level_0','Dates')], errors = 'coerce')
    raw_df = raw_df.drop(('Unnamed: 0_level_0','Dates'), axis = 1)
    
    ##-------check any wrong date format--------##
    print(f'sum of NaN in dates = {raw_df[('real_date','real_date')].isna().sum()}')

    ##----------sort datetime----------#
    raw_df = raw_df.sort_values(('real_date', 'real_date'))
    
    ##------drop rows with duplicate date and ticker--------##
    raw_df = raw_df.drop_duplicates(subset = [('real_date', 'real_date')])
    
    ##-------set date as index---------#
    raw_df = raw_df.set_index(('real_date', 'real_date'))
    
    #-------Drop original date---------#
    # raw_df = raw_df.drop(("Unnamed: 0_level_0","Dates"), axis = 1)
    
    return raw_df

In [5]:
df = clean_data_date(df)

sum of NaN in dates = 316


In [6]:
##-------3. change_to_long data --------##
    
def change_to_long(raw_df):
    return raw_df.stack(level = 0)

In [7]:
df = change_to_long(df)
# print(df.shape)
print('LONGGGGG')
print(df.head())

LONGGGGG
                                             PX_LAST  PX_VOLUME  PX_OPEN  \
(real_date, real_date)                                                     
2023-01-02             TOY_EQUITY US Equity      NaN        NaN      NaN   
2023-01-03             TOY_EQUITY US Equity    96.63   718868.0    97.11   
2023-01-05             TOY_EQUITY US Equity    85.31   135494.0    85.74   
2023-01-06             TOY_EQUITY US Equity      NaN        NaN      NaN   
2023-01-08             TOY_EQUITY US Equity      NaN        NaN      NaN   

                                             PX_LOW  PX_HIGH  
(real_date, real_date)                                        
2023-01-02             TOY_EQUITY US Equity     NaN      NaN  
2023-01-03             TOY_EQUITY US Equity   95.66    97.60  
2023-01-05             TOY_EQUITY US Equity   84.46    86.16  
2023-01-06             TOY_EQUITY US Equity     NaN      NaN  
2023-01-08             TOY_EQUITY US Equity     NaN      NaN  


In [8]:
##-------4. check type mismatch --------##

def check_type(long_raw_df):
    cols = ['PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST', 'PX_VOLUME']
    original_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    long_raw_df[cols] = long_raw_df[cols].apply(pd.to_numeric, errors = 'coerce')
    
    after_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    ##-----NaN difference between original and after--------#
    print(original_null - after_null)
    
    long_raw_df.index = long_raw_df.index.set_names(['real_date', 'ticker'])
    return long_raw_df

In [9]:
df = check_type(df)

PX_OPEN      0
PX_HIGH      0
PX_LOW       0
PX_LAST      0
PX_VOLUME    0
dtype: int64


In [13]:
##-------5. drop non universal ticker --------##
def drop_non_universal(long_raw_df):
      #------------PRICE--------------#
    #1.check if it is under universal 100 in that year
    check_universal = long_raw_df.copy()

    check_universal = check_universal.reset_index()
    
    check_universal['year'] = check_universal['real_date'].dt.year
    
    ##proportion of non-null/all < 0.5 -> drop
    check_universal = check_universal.groupby(['ticker', 'year']).apply(lambda x: x.count()/(x.count()+x.isnull().sum())).map(lambda x: x<0.5)
    
    check_universal['to_drop'] = check_universal.any(axis = 1)
    
    check_universal = check_universal.reset_index()
    
    to_drop_list = check_universal[check_universal['to_drop']]['ticker'].tolist()
    
    print(to_drop_list)
    
    long_raw_df = long_raw_df[~long_raw_df.index.get_level_values('ticker').isin(to_drop_list)]
    
    return long_raw_df

In [14]:
# print("\n--- 5. Drop non-universal tickers ---")
before = df.index.get_level_values('ticker').nunique()
df = drop_non_universal(df)
after = df.index.get_level_values('ticker').nunique()
print(f"Tickers: {before} -> {after}")

['TOY_EQUITY US Equity']
Tickers: 1 -> 0


In [15]:
##-------6. check misalign date --------##

def check_misalign_date(long_raw_data):
    long_raw_data = long_raw_data.reset_index()

    date_counts = long_raw_data.groupby('real_date')['ticker'].count()
    correct_date =  date_counts[date_counts == date_counts.max()]
    
    correct_date = correct_date.index.tolist()
    
    long_raw_data = long_raw_data[long_raw_data['real_date'].isin(correct_date)]
    
    long_raw_data = long_raw_data.set_index(['real_date', 'ticker'])
    
    return long_raw_data

In [16]:
# print("\n--- 6. Fix misaligned dates ---")
before = df.shape[0]
df = check_misalign_date(df)
after = df.shape[0]
# print(f"Rows: {before} -> {after}")

In [17]:
##-------7. track unusual price and volume --------##

def check_price_and_volume(long_raw_df):
    # print(long_raw_df)
    long_raw_df.loc[lambda x: ~((x['PX_LOW'] < x['PX_LAST']) & (x['PX_LAST']  < x['PX_HIGH']) & (x['PX_LOW'] < x['PX_OPEN']) & (x['PX_OPEN']< x['PX_HIGH'])), ['PX_LOW', 'PX_HIGH', 'PX_OPEN', 'PX_LAST']] = None
    ##---------Track unusual volume with Z-score------------##
    
    mean = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('mean')
    std = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('std')
    long_raw_df['z_score'] = (long_raw_df['PX_VOLUME'] - mean) / std
    long_raw_df.loc[lambda x: x['z_score'].abs() > 3, 'PX_VOLUME'] = None
    long_raw_df = long_raw_df.drop('z_score', axis = 1)
    
    long_raw_df = long_raw_df.groupby('ticker').ffill(limit = 3)
    
    return long_raw_df

In [ ]:
# print("\n--- 7. Check price/volume anomalies ---")
df = check_price_and_volume(df)

In [ ]:
##-------8. add return columns --------##
def add_return_columns(aligned_df):
    """
    Input : aligned price DataFrame
    Output: same DataFrame + simple return and log return columns
            (basic derived series only — no risk/strategy metrics here)
    """
    aligned_df['simple_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform('pct_change')
    
    aligned_df['log_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform(lambda x : np.log(x/x.shift(1)))
    
    aligned_df[['simple_return', 'log_return']] = aligned_df[['simple_return', 'log_return']].fillna(0)
    
    return aligned_df

In [ ]:
# print("\n--- 8. Add return columns ---")
df = add_return_columns(df)
# print(df[['PX_LAST', 'simple_return', 'log_return']].head())

In [ ]:
def validate_data(final_df):
    """
    Input : fully processed DataFrame
    Output: pass/fail or list of issues —
            checks for leftover NaNs, mismatched row counts,
            out-of-range values, unexpected date gaps
    """
    issues = []
    
    null_counts = final_df.isnull().sum()
    if null_counts.sum() > 0:
        issues.append(f"NaNs remaining (expected due to ffill limit=3):\n{null_counts[null_counts > 0]}")
    
    row_counts = final_df.groupby('ticker').size()
    if row_counts.nunique() > 1:
        issues.append(f"Mismatched row counts:\n{row_counts}")
    
    if (final_df[['PX_LAST','PX_OPEN','PX_HIGH','PX_LOW']] < 0).any().any():
        issues.append("Negative prices found")
    
    if issues:
        for i in issues:
            print(i)
    else:
        print("All checks passed")
        

In [ ]:
# print("\n--- 9. Validate final data ---")
issues = validate_data(df)